<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_customer_churn_nb1_enonce.png" width="100%"/>
</div>

# Notebook 1 — SQL Analytics : Customer Churn Analytics — *Énoncé*

## Contexte métier

**IvoirCom** est un opérateur télécom mobile fictif basé à Abidjan, opérant dans 5 villes ivoiriennes (Abidjan, Bouaké, Yamoussoukro, San-Pédro, Korhogo). Sa direction commerciale alerte : **12 % de la base abonnés churn chaque trimestre**, soit un coût de rétention/acquisition (CAC perdu) estimé à 85 000 FCFA par départ.

Tu joues le rôle d'analyste data fraîchement embauché. Mission :

1. **Comprendre** la composition de la base (volumétrie, segments).
2. **Mesurer** le taux de churn par offre, ville, tranche d'âge.
3. **Identifier** les signaux avant-coureurs (consommation en baisse, réclamations non résolues).
4. **Segmenter** les abonnés via RFM (Récence facturation, Fréquence réclamations, Monétaire ARPU).
5. **Recommander** 3 leviers d'action chiffrés à la direction commerciale.

## Données disponibles

| Table | Lignes | Description |
|---|---|---|
| `clients` | 8 030 | Abonnés (id, ville, offre, date souscription, statut) — contient 30 doublons et 5 âges négatifs |
| `offres` | 6 | Catalogue offres (Pulse, Connect, Premium, Pro, Étudiant, Senior) |
| `factures` | 138 284 | Facturation mensuelle 24 mois (2 % de montants nuls) |
| `consommation_mensuelle` | 137 044 | Voix / SMS / Data par client par mois |
| `reclamations` | 9 791 | Tickets support (3 % de délais de résolution négatifs) |

## Outils

- **DuckDB** + **JupySQL** (`%%sql` magic) — analytique SQL en mémoire, lecture CSV native.
- **pandas** — manipulation tabulaire complémentaire et visualisations.
- **matplotlib / seaborn** — graphiques.

## Plan du notebook (7 sections)

| # | Section | Concepts SQL clés |
|---|---|---|
| 0 | Setup & chargement | `read_csv_auto`, `CREATE TABLE` |
| 1 | Exploration & nettoyage | `FILTER`, `CAST`, `ABS`, `CASE WHEN`, `CREATE VIEW` |
| 2 | KPIs globaux churn | `COUNT FILTER`, `AVG`, `MEDIAN`, `DATE_DIFF` |
| 3 | Segmentation | `JOIN`, `GROUP BY`, heatmap |
| 4 | Cohortes | `DATE_TRUNC`, `generate_series`, pivot |
| 5 | Signaux Réclamations | window `SUM() OVER`, `HAVING` |
| 6 | RFM Telecom | `NTILE`, `CTE` chaînées |
| 7 | Synthèse & recommandations | tableau + 3 leviers chiffrés |

## Comment travailler avec ce notebook

| Symbole | Ce que ça veut dire |
|---|---|
| 🎯 | **Objectif** — ce que tu dois produire dans la cellule suivante |
| 🔧 | **Méthode** — l'intuition technique avant de coder |
| 🤔 | **Questions de réflexion** — à formuler après l'exécution, en français, dans une cellule markdown ajoutée |
| 🏥 | **Métier** — la conséquence pour la direction commerciale |
| ⚠️ | **Piège** courant à éviter |

---
## 0. Setup — imports, paramétrage, connexion DuckDB

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine seaborn --quiet

### 🔧 MÉTHODE — pourquoi DuckDB plutôt que pandas seul ?

DuckDB exécute du SQL pur sur des CSV directement, sans charger d'ETL préalable. Il gère les fenêtres analytiques (`RANK`, `NTILE`, `LAG`), les `CTE` chaînées, et les agrégations multi-grain bien plus lisibles qu'en pandas. JupySQL relie DuckDB aux cellules `%%sql` du notebook tout en exposant les résultats comme des `DataFrame` pandas pour la visualisation.

### 🎯 Objectif — Cellule suivante

Importer les bibliothèques nécessaires (`pandas`, `numpy`, `matplotlib`, `seaborn`, `duckdb`), définir le dictionnaire `COLORS` de la charte DataProjectLab et configurer matplotlib pour des graphiques cohérents avec le reste des notebooks.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import duckdb
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
    "light":     "#EEEDFE",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})  

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/logitrack_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅') 

### 🔧 MÉTHODE — chargement des CSV en tables DuckDB

`read_csv_auto` détecte automatiquement les types et les en-têtes. On charge les 5 CSV sources (clients, offres, factures, consommation, reclamations) en tables `_raw` (brut) avant nettoyage. Pour les colonnes date, on créera plus loin des **vues typées** avec `CAST(col AS DATE)`.

### 🎯 Objectif — Cellule suivante

1. Créer 5 tables DuckDB (`clients_raw`, `offres`, `factures_raw`, `consommation`, `reclamations_raw`) depuis les 5 CSV.
2. Vérifier que toutes les tables sont chargées en imprimant la volumétrie de chacune en une seule requête `UNION ALL`.

In [ ]:
BASE_URL   = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_churn_analytics/data/'


conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE clients_raw AS SELECT * FROM read_csv_auto('{BASE_URL}clients.csv');
    CREATE TABLE offres     AS SELECT * FROM read_csv_auto('{BASE_URL}offres.csv');
    CREATE TABLE factures_raw        AS SELECT * FROM read_csv_auto('{BASE_URL}factures.csv');
    CREATE TABLE consommation AS SELECT * FROM read_csv_auto('{BASE_URL}consommation_mensuelle.csv');
    CREATE TABLE reclamations_raw AS SELECT * FROM read_csv_auto('{BASE_URL}reclamations.csv');
""")

n = conn.execute('SELECT COUNT(*) FROM consommation').fetchone()[0]
print(f'✅ {n:,} livraisons chargées dans DuckDB')

%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

In [ ]:
%%sql

SELECT 'clients' AS table_name, COUNT(*) AS n FROM clients_raw
UNION ALL SELECT 'offres', COUNT(*) FROM offres
UNION ALL SELECT 'factures', COUNT(*) FROM factures_raw
UNION ALL SELECT 'consommation', COUNT(*) FROM consommation
UNION ALL SELECT 'reclamations', COUNT(*) FROM reclamations_raw;

### 🤔 Questions de réflexion (à répondre dans une cellule markdown que tu ajoutes)

1. Quel est le ratio `factures / clients_distincts` ? Que dit-il sur la profondeur d'historique de la base ?
2. Quel est le ratio `réclamations / clients` ? Est-ce qu'on peut déjà supposer que tous les clients se plaignent autant, ou qu'une frange concentre les plaintes ?
3. Pourquoi la table `consommation` a-t-elle moins de lignes que `factures` (137 044 vs 138 284) ?

---
## 1. Exploration & nettoyage des anomalies

Avant tout calcul de KPI, on inventorie et corrige les défauts qualité connus. Une donnée non nettoyée fausse silencieusement les indicateurs (un âge négatif décale les segments, un montant nul tire l'ARPU vers le bas, un délai négatif rend la médiane absurde).

### 1.1 Doublons clients (suffixe `_DUP`) et âges négatifs

#### 🔧 MÉTHODE

Le générateur a inséré 30 doublons (id_client se terminant par `_DUP`) et 5 âges négatifs (signe inversé). Tu vas d'abord les **compter**, puis créer une **vue `clients`** propre qui exclut les doublons et applique `ABS(age)`.

#### 🎯 Objectif — Cellule suivante

Compter en une seule requête : nombre de doublons (id finissant par `_DUP`), nombre d'âges négatifs, total brut. Utiliser la syntaxe `COUNT(*) FILTER (WHERE ...)`.

In [ ]:
%%sql
-- TODO : SELECT COUNT(*) FILTER (WHERE id_client LIKE '%_DUP') AS doublons_dup,
--               COUNT(*) FILTER (WHERE age < 0)                AS ages_negatifs,
--               COUNT(*)                                       AS total_brut
--        FROM clients_raw




### 🎯 Objectif — Cellule suivante

Créer une vue `clients` qui :

1. Exclut les doublons (`WHERE id_client NOT LIKE '%_DUP'`)
2. Corrige l'âge avec `ABS(age)`
3. Type les colonnes date avec `CAST(... AS DATE)` (sur `date_souscription` et `date_resiliation`)

Vérifier que la vue contient bien 8 000 lignes.

In [ ]:
%%sql
-- TODO 1 : CREATE OR REPLACE VIEW clients AS SELECT ...
--          colonnes : id_client, nom, prenom, sexe, ABS(age) AS age, ville,
--                     code_offre, CAST(date_souscription AS DATE) AS date_souscription,
--                     CAST(date_resiliation AS DATE) AS date_resiliation, statut
--          FROM clients_raw
--          WHERE id_client NOT LIKE '%_DUP'
-- TODO 2 : SELECT COUNT(*) AS clients_propres FROM clients;
--          attendu : 8 000




### 1.2 Montants nuls dans `factures`

#### 🔧 MÉTHODE

2 % des factures ont un montant à 0 — soit erreur de saisie, soit promo gratuite. On les **exclut** du calcul d'ARPU mais on les **conserve** pour la volumétrie. Vue `factures` typée pour la suite.

#### 🎯 Objectif — 2 cellules suivantes

1. Compter le total et les montants nuls + le pourcentage.
2. Créer la vue `factures` typée (cast date et montant en INTEGER).

In [ ]:
%%sql
-- TODO : SELECT COUNT(*), COUNT(*) FILTER (WHERE montant_fcfa = 0),
--               ROUND(100.0 * COUNT(*) FILTER (WHERE montant_fcfa = 0) / COUNT(*), 2) AS pct_nuls
--        FROM factures_raw




In [ ]:
%%sql
-- TODO : CREATE OR REPLACE VIEW factures AS
--        SELECT id_facture, id_client,
--               CAST(date_facturation AS DATE) AS date_facturation,
--               CAST(montant_fcfa AS INTEGER) AS montant_fcfa,
--               methode_paiement
--        FROM factures_raw




### 1.3 Délais de résolution négatifs (`reclamations`)

#### 🔧 MÉTHODE

Un délai négatif (résolu *avant* d'être ouvert) est physiquement impossible — c'est un bug de saisie côté centre d'appels. On le corrige par `ABS()`. La vue `reclamations` typée date est créée.

#### 🎯 Objectif — 2 cellules suivantes

1. Compter le total, les délais négatifs, les délais NULL.
2. Créer la vue `reclamations` typée date avec délai corrigé via `CASE WHEN delai < 0 THEN ABS(delai) ELSE delai END`.

In [ ]:
%%sql
-- TODO : SELECT COUNT(*) AS total,
--               COUNT(*) FILTER (WHERE delai_resolution_jours < 0)     AS delais_negatifs,
--               COUNT(*) FILTER (WHERE delai_resolution_jours IS NULL) AS delais_null
--        FROM reclamations_raw




In [ ]:
%%sql
-- TODO : CREATE OR REPLACE VIEW reclamations AS
--        SELECT id_ticket, id_client,
--               CAST(date_creation AS DATE) AS date_creation,
--               type, statut,
--               CASE WHEN delai_resolution_jours < 0
--                    THEN ABS(delai_resolution_jours)
--                    ELSE delai_resolution_jours
--               END AS delai_resolution_jours
--        FROM reclamations_raw




### 🤔 Questions de réflexion

1. Combien de tickets restent en `ouvert` (délai NULL) ? Que pourrait dire ce chiffre sur la **saturation du support client** ?
2. Pourquoi traite-t-on différemment les âges négatifs (correction par `ABS`) et les doublons (suppression) ?
3. Quelle anomalie aurait l'impact le plus grave sur l'ARPU si on ne nettoyait pas (montants nuls, doublons clients, délais négatifs) ? Pourquoi ?

---
## 2. KPIs globaux du churn

On commence par les chiffres qui apparaissent en page 1 du dashboard exécutif : taux de churn cumulatif sur la période, ARPU global, anciennetés comparées.

### 2.1 Taux de churn cumulatif et répartition statut

#### 🔧 MÉTHODE

Le **taux de churn cumulatif** = `nb_resilies / nb_total`. C'est la part de clients qui ont quitté IvoirCom à un moment quelconque sur les 24 mois observés. À ne pas confondre avec le **taux de churn trimestriel** (12 % annoncé par la DG) qui se calcule sur une fenêtre glissante de 90 jours.

#### 🎯 Objectif — Cellule suivante

Calculer le nombre de clients par `statut` (`actif` / `resilie`) et leur pourcentage dans la base. Utiliser la window function `SUM(COUNT(*)) OVER ()` pour le pourcentage.

In [ ]:
%%sql
-- TODO : SELECT statut, COUNT(*) AS n_clients,
--               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
--        FROM clients GROUP BY statut ORDER BY n_clients DESC




### 2.2 ARPU global, médiane et écart actifs vs churners

#### 🔧 MÉTHODE

L'**ARPU** (Average Revenue Per User) se calcule sur les factures non nulles. La médiane est plus robuste que la moyenne pour une distribution asymétrique. L'écart actifs vs churners est un *signal de valeur perdue*.

#### 🎯 Objectif — Cellule suivante

Calculer pour chaque statut (`actif` / `resilie`) :
- L'ARPU mensuel **moyen** par client (par client puis moyenné)
- L'ARPU mensuel **médian**
- Le min et le max

*Hint* : utiliser une `CTE` qui calcule d'abord `arpu_mensuel = AVG(montant_fcfa)` par client (en filtrant `WHERE montant > 0`), puis joindre avec `clients` pour grouper par statut.

In [ ]:
%%sql
-- TODO : WITH arpu_par_client AS (
--          SELECT id_client, AVG(montant_fcfa) AS arpu_mensuel
--          FROM factures WHERE montant_fcfa > 0 GROUP BY id_client
--        )
--        SELECT c.statut, COUNT(*) AS n_clients,
--               ROUND(AVG(a.arpu_mensuel), 0)    AS arpu_moyen_fcfa,
--               ROUND(MEDIAN(a.arpu_mensuel), 0) AS arpu_median_fcfa,
--               ROUND(MIN(a.arpu_mensuel), 0), ROUND(MAX(a.arpu_mensuel), 0)
--        FROM clients c JOIN arpu_par_client a USING (id_client)
--        GROUP BY c.statut




### 2.3 Ancienneté moyenne actifs vs churners

#### 🔧 MÉTHODE

L'ancienneté = `date_resiliation - date_souscription` pour les churners, `date_pivot - date_souscription` pour les actifs. On utilise `'2025-12-31'` comme date pivot pour les actifs afin de figer la comparaison.

#### 🎯 Objectif — Cellule suivante

Calculer la moyenne et la médiane d'ancienneté en **mois** par statut. Utiliser `DATE_DIFF('day', date_souscription, COALESCE(date_resiliation, DATE '2025-12-31')) / 30.0`.

In [ ]:
%%sql
-- TODO : SELECT statut, COUNT(*),
--               ROUND(AVG(DATE_DIFF('day', date_souscription,
--                         COALESCE(date_resiliation, DATE '2025-12-31')) / 30.0), 1)
--                 AS anciennete_moyenne_mois,
--               ROUND(MEDIAN(... meme calcul ...), 1) AS anciennete_mediane_mois
--        FROM clients GROUP BY statut




### 🤔 Questions de réflexion

1. **Quel est le taux de churn cumulatif** ? La DG parle de 12 % par trimestre — ce chiffre est-il cohérent avec le cumulatif sur 24 mois ?
2. **L'écart d'ARPU actifs vs churners est-il faible (< 20 %) ou massif (> 50 %)** ? Que conclus-tu sur le **profil des churners** (bas de gamme ou premium) ?
3. **Les churners partent-ils tôt ou tard** dans leur cycle de vie ? Que dit cela sur la qualité de l'onboarding ?

> 🏥 **MÉTIER** — Si les churners ont un ARPU significativement inférieur, IvoirCom perd surtout de la **base bas de gamme** (offres Pulse, Étudiant). C'est moins critique financièrement mais ça érode la part de marché. Si l'écart est faible, le churn touche aussi les offres premium — il faut alors prioritairement la rétention.

---
## 3. Segmentation : qui churne le plus ?

On éclate le taux de churn cumulatif par dimensions opérationnelles : **offre**, **ville**, **tranche d'âge**. L'objectif est d'identifier 2-3 segments hautement actionnables.

### 3.1 Taux de churn par offre

#### 🔧 MÉTHODE

Une jointure `clients ⋈ offres` permet d'enrichir avec le libellé et le prix. Le taux de churn par offre = `nb_resilies / nb_total` *au sein* de chaque offre. On trie descendant pour faire émerger les offres « porte-tournante ».

#### 🎯 Objectif — 2 cellules suivantes

1. Calculer pour chaque offre : `code_offre`, `libelle`, `prix_fcfa`, `n_total`, `n_churners`, `taux_churn_pct`. Trier descendant par taux.
2. Tracer un graphique en **barres horizontales** des taux de churn par offre, avec une ligne pointillée à 25 % (seuil critique).

In [ ]:
%%sql
-- TODO : SELECT o.code_offre, o.libelle, o.prix_fcfa,
--               COUNT(*) AS n_total,
--               COUNT(*) FILTER (WHERE c.statut = 'resilie') AS n_churners,
--               ROUND(100.0 * COUNT(*) FILTER (...) / COUNT(*), 1) AS taux_churn_pct
--        FROM clients c JOIN offres o USING (code_offre)
--        GROUP BY o.code_offre, o.libelle, o.prix_fcfa
--        ORDER BY taux_churn_pct DESC




In [ ]:
# TODO 1 : recuperer le resultat de la requete dans df_offre via
#          df_offre = %sql SELECT ...
# TODO 2 : tracer un barh avec axe Y = libelle, axe X = taux_churn_pct, color=COLORS['primary']
# TODO 3 : ajouter axvline a 25 (couleur danger, --) avec label 'Seuil critique 25 %'
# TODO 4 : ajouter titre, xlabel, legend, invert_yaxis (pour avoir le pire en haut)
# TODO 5 : plt.tight_layout() puis plt.show()




### 3.2 Taux de churn par ville et par tranche d'âge

#### 🔧 MÉTHODE

On découpe l'âge en tranches métier (18-25, 26-35, 36-45, 46-60, 60+). En SQL, on utilise `CASE WHEN ... BETWEEN ... THEN ...`.

#### 🎯 Objectif — 2 cellules suivantes

1. Calculer `n_total`, `n_churners`, `taux_churn_pct` par ville. Trier descendant par taux.
2. Idem pour les 5 tranches d'âge.

In [ ]:
%%sql
-- TODO : taux de churn par ville, trie descendant




In [ ]:
%%sql
-- TODO : utiliser CASE WHEN age BETWEEN 18 AND 25 THEN '1. 18-25'
--                       WHEN age BETWEEN 26 AND 35 THEN '2. 26-35'
--                       ...
--                       ELSE '5. 60+' END AS tranche_age




### 3.3 Heatmap croisée Offre × Ville (taux de churn)

#### 🔧 MÉTHODE

Une heatmap est l'outil idéal pour repérer une **interaction** : par exemple, l'offre Pulse churn-t-elle particulièrement à Bouaké ? On utilise `seaborn.heatmap` avec une palette divergente centrée sur le taux moyen (~25 %).

#### 🎯 Objectif — Cellule suivante

1. Récupérer une table `(ville, code_offre, taux_churn)` via `%sql`.
2. Pivoter en table 2D `ville × code_offre`.
3. Tracer la heatmap avec `cmap='RdYlGn_r'` (rouge = mauvais, vert = bon), `center=25`, et annotations à 1 décimale.

In [ ]:
# TODO 1 : df_cross = %sql SELECT ville, code_offre,
#                                ROUND(100.0 * COUNT(*) FILTER (WHERE statut = 'resilie')
#                                       / COUNT(*), 1) AS taux_churn
#                          FROM clients GROUP BY ville, code_offre;
# TODO 2 : pivot = df_cross.pivot(index='ville', columns='code_offre', values='taux_churn')
# TODO 3 : sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r', center=25, ...)




### 🤔 Questions de réflexion

1. **Quelles sont les 2 offres avec le taux de churn le plus élevé** ? Que peux-tu déduire sur leur positionnement (entrée de gamme, premium) ?
2. **L'écart entre les villes est-il important** (> 10 pp) ou faible (< 5 pp) ? Le churn est-il plutôt **géographique** ou **comportemental** ?
3. Dans la heatmap, **identifie les 2 cellules les plus rouges** (offre × ville) — quelles actions opérationnelles sont déclenchées par ce constat ?
4. Quelle tranche d'âge churne le plus ? Cet écart est-il actionnable (campagne ciblée par âge) ou marginal ?

> 🏥 **MÉTIER** — Une offre d'entrée de gamme avec un churn > 30 % est typiquement un signal qu'elle ne **convertit pas vers le haut** (les utilisateurs essaient puis quittent au lieu de migrer vers une offre supérieure). Une ville secondaire avec un churn élevé peut signaler un déficit de couverture réseau.

---
## 4. Analyse de cohortes — quelle génération de souscripteurs tient le mieux ?

Une **cohorte** = ensemble des clients souscrits le même mois. La courbe de rétention par cohorte révèle si la dégradation du churn est **structurelle** (toutes les cohortes pareilles) ou **cyclique** (certaines cohortes chutent vite à cause d'un événement).

### 4.1 Construction de la table de cohortes

#### 🔧 MÉTHODE

Pour chaque (cohorte, période d'observation), on calcule la part des clients de la cohorte encore actifs à cette période. Logique :

1. `cohorte_mois` = `DATE_TRUNC('month', date_souscription)`
2. Pour chaque mois M depuis la souscription, flag *actif* si `date_resiliation` n'est pas encore atteinte ou est NULL.
3. Agréger par `(cohorte_mois, mois_depuis_souscription)`.

On limite aux cohortes **2024** (12 cohortes) pour avoir au moins 12 mois d'observation.

#### 🎯 Objectif — Cellule suivante

Créer une vue `cohortes(cohorte_mois, mois_depuis_souscription, clients_actifs)` en utilisant `generate_series` pour étendre chaque client mois par mois entre `date_souscription` et `date_resiliation` (ou 2025-12-31 si NULL).

In [ ]:
%%sql
-- TODO : CREATE OR REPLACE VIEW cohortes AS
--        WITH base AS (
--          SELECT id_client,
--                 DATE_TRUNC('month', date_souscription) AS cohorte_mois,
--                 date_souscription,
--                 COALESCE(date_resiliation, DATE '2025-12-31') AS date_fin
--          FROM clients WHERE EXTRACT(YEAR FROM date_souscription) = 2024
--        ),
--        expand AS (
--          SELECT b.id_client, b.cohorte_mois,
--                 DATE_DIFF('month', b.cohorte_mois, g.gen_date) AS mois_depuis_souscription
--          FROM base b,
--               generate_series(b.date_souscription, b.date_fin, INTERVAL 1 MONTH) AS g(gen_date)
--        )
--        SELECT cohorte_mois, mois_depuis_souscription,
--               COUNT(DISTINCT id_client) AS clients_actifs
--        FROM expand WHERE mois_depuis_souscription BETWEEN 0 AND 11
--        GROUP BY cohorte_mois, mois_depuis_souscription
--        ORDER BY cohorte_mois, mois_depuis_souscription;
-- Puis : SELECT * FROM cohortes LIMIT 24;




### 4.2 Pivot et heatmap de rétention

#### 🔧 MÉTHODE

On pivote en table 12×12 : lignes = cohorte, colonnes = mois écoulé. Chaque cellule = % de clients de la cohorte encore actifs. Le mois 0 est toujours 100 %, puis la courbe descend.

#### 🎯 Objectif — Cellule suivante

1. Récupérer la vue cohortes en DataFrame.
2. Calculer `taux_retention_pct` = `clients_actifs / size_at_M0 * 100`.
3. Pivoter en table 2D et tracer une heatmap avec `cmap='RdYlGn'` (vert = bonne rétention) et `vmin=50, vmax=100`.

In [ ]:
# TODO 1 : df_coh = %sql SELECT * FROM cohortes;
# TODO 2 : convertir cohorte_mois en datetime + creer cohorte_label (format YYYY-MM)
# TODO 3 : extraire size_at_0 (taille de chaque cohorte au mois 0)
# TODO 4 : calculer df_coh['taux_retention_pct'] = 100 * clients_actifs / size_at_0[label]
# TODO 5 : pivot = df_coh.pivot(index='cohorte_label', columns='mois_depuis_souscription',
#                               values='taux_retention_pct')
# TODO 6 : sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn', vmin=50, vmax=100, ...)




### 🤔 Questions de réflexion

1. **La rétention M+3 est-elle supérieure à 90 %** ? Si oui, l'onboarding fonctionne bien — sinon, il y a un problème de premier mois.
2. **La dégradation est-elle régulière** (rétention descend lentement de M+3 à M+12) ou **brutale** (chute soudaine à un mois précis) ?
3. **Identifie une cohorte qui se distingue** — quel pourrait être l'événement opérationnel à l'origine (lancement nouvelle offre concurrente, panne réseau, changement tarifaire) ?

> 🏥 **MÉTIER** — Si la rétention M+3 est inférieure à 90 %, le problème est dans l'**onboarding**. Si la dégradation est régulière de M+3 à M+12, c'est un problème **structurel** (offre commoditisée, prix, concurrence).

---
## 5. Signaux faibles — les réclamations prédisent-elles le départ ?

Hypothèse métier : un client qui ouvre un ticket et ne reçoit pas de résolution (ticket *abandonné* ou *ouvert > 30j*) part dans les semaines qui suivent. On va le **vérifier en SQL**.

### 5.1 Top types de réclamations et statut de résolution

#### 🔧 MÉTHODE

Croisement `type × statut` avec `COUNT(*)` et **% des lignes par type** via window function `SUM(COUNT(*)) OVER (PARTITION BY type)`.

#### 🎯 Objectif — Cellule suivante

Pour chaque combinaison `(type, statut)`, calculer le nombre et le **pourcentage au sein du type** (la somme doit faire 100 % par type).

In [ ]:
%%sql
-- TODO : SELECT type, statut, COUNT(*) AS n,
--               ROUND(100.0 * COUNT(*) /
--                     SUM(COUNT(*)) OVER (PARTITION BY type), 1) AS pct_dans_type
--        FROM reclamations GROUP BY type, statut
--        ORDER BY type, n DESC




### 5.2 Comparaison nb tickets churners vs actifs

#### 🔧 MÉTHODE

On compte le nombre de tickets par client, on joint avec `clients` pour récupérer le statut, puis on calcule la moyenne par groupe. Si les churners ont en moyenne 2x plus de tickets que les actifs, c'est confirmé que la frustration mène au départ.

#### 🎯 Objectif — Cellule suivante

Calculer pour chaque statut client :
- nombre de clients
- tickets moyens par client (avec `COALESCE` pour gérer les clients sans ticket)
- abandonnés moyens, ouverts moyens
- pourcentage de clients ayant 2+ tickets

In [ ]:
%%sql
-- TODO : WITH tickets_par_client AS (
--          SELECT id_client,
--                 COUNT(*) AS n_tickets,
--                 COUNT(*) FILTER (WHERE statut = 'abandonne') AS n_abandonnes,
--                 COUNT(*) FILTER (WHERE statut = 'ouvert')    AS n_ouverts
--          FROM reclamations GROUP BY id_client
--        )
--        SELECT c.statut AS statut_client, COUNT(*) AS n_clients,
--               ROUND(AVG(COALESCE(t.n_tickets, 0)), 2)    AS tickets_moyens,
--               ROUND(AVG(COALESCE(t.n_abandonnes, 0)), 2) AS abandonnes_moyens,
--               ROUND(AVG(COALESCE(t.n_ouverts, 0)), 2)    AS ouverts_moyens,
--               ROUND(100.0 * COUNT(*) FILTER (WHERE t.n_tickets >= 2)
--                     / COUNT(*), 1) AS pct_avec_2plus_tickets
--        FROM clients c LEFT JOIN tickets_par_client t USING (id_client)
--        GROUP BY c.statut




### 5.3 Test du signal : taux de churn parmi clients à 2+ tickets non résolus

#### 🔧 MÉTHODE

On isole les clients qui ont au moins 2 tickets en statut *ouvert* ou *abandonné* (= signal d'insatisfaction prolongée). On compare leur taux de churn à la base globale via un `UNION ALL`. **Si l'écart est > 2x, le signal est fort et opérationnel.**

#### 🎯 Objectif — Cellule suivante

Calculer en une seule requête (via `UNION ALL`) :
1. Le nombre de clients à 2+ tickets non résolus + leur taux de churn.
2. Le total de la base + son taux de churn global.

In [ ]:
%%sql
-- TODO : WITH clients_a_risque AS (
--          SELECT id_client FROM reclamations
--          WHERE statut IN ('ouvert', 'abandonne')
--          GROUP BY id_client HAVING COUNT(*) >= 2
--        )
--        SELECT 'Clients a 2+ tickets non resolus' AS segment,
--               COUNT(*), COUNT(*) FILTER (WHERE c.statut = 'resilie'),
--               ROUND(100.0 * COUNT(*) FILTER (...) / COUNT(*), 1) AS taux_churn_pct
--        FROM clients c JOIN clients_a_risque USING (id_client)
--        UNION ALL
--        SELECT 'Base globale', COUNT(*), COUNT(*) FILTER (...),
--               ROUND(100.0 * COUNT(*) FILTER (...) / COUNT(*), 1)
--        FROM clients




### 🤔 Questions de réflexion

1. **Quel est le top type de réclamation** ? Que dit-il sur le **maillon faible** d'IvoirCom (réseau, facturation, offre, matériel) ?
2. **Quel est le ratio entre tickets churners et tickets actifs** ? > x2 confirme l'hypothèse métier — < x2 invalide.
3. **Quel est le taux de churn parmi les clients à 2+ tickets non résolus** ? Compare-le à la base globale.
4. Si tu devais lancer une **cellule rétention dédiée**, combien de conseillers faudrait-il pour traiter le segment « 2+ tickets non résolus » à raison de 5 appels par jour par conseiller (sur 1 mois) ?

> 🏥 **MÉTIER** — Le signal *« 2+ tickets non résolus = clients à appeler aujourd'hui »* est le levier de rétention le plus rapide à mettre en place : il ne nécessite pas de modèle ML, juste un script SQL hebdomadaire et 5-10 conseillers dédiés.

---
## 6. Segmentation RFM Telecom — qui sont mes Champions, mes At-Risk, mes Lost ?

Le **RFM télécom** adapte la grille e-commerce :

- **R** (Récence) : jours depuis la dernière facture ; petit = récent = bon.
- **F** (Fréquence) : *inversée* — nb de réclamations sur 6 mois ; petit = peu de plaintes = bon.
- **M** (Monétaire) : ARPU des 6 derniers mois ; grand = bon.

On `NTILE(5)` chaque dimension (1 = pire, 5 = meilleur) puis on combine en segments lisibles.

### 6.1 Calcul des 3 dimensions RFM

#### 🔧 MÉTHODE

Date pivot = `2025-12-31`. Fenêtre 6 derniers mois = `[2025-07-01, 2025-12-31]`. On joint 3 sous-requêtes (`recence`, `frequence`, `monetaire`) à `clients`, on applique `NTILE(5)` partition par dimension.

#### 🎯 Objectif — Cellule suivante

Créer une table `rfm` qui contient pour chaque client :
- `jours_depuis_derniere_facture` (depuis 2025-12-31)
- `n_reclamations_6m` (depuis 2025-07-01)
- `arpu_6m` (moyenne factures 2025-07-01+, montant > 0)
- `R_score`, `F_score`, `M_score` via `NTILE(5)` (attention au sens du tri !)

In [ ]:
%%sql
-- TODO : CREATE OR REPLACE TABLE rfm AS
--        WITH date_pivot AS (SELECT DATE '2025-12-31' AS d_pivot),
--             recence AS ( ... DATE_DIFF('day', MAX(date_facturation), d_pivot) ... ),
--             frequence AS ( ... COUNT(*) FROM reclamations WHERE date_creation >= '2025-07-01' ... ),
--             monetaire AS ( ... AVG(montant_fcfa) FROM factures WHERE date >= '2025-07-01' AND > 0 ... )
--        SELECT c.id_client, c.code_offre, c.ville, c.statut,
--               COALESCE(...) AS jours_depuis_derniere_facture,
--               COALESCE(...) AS n_reclamations_6m,
--               COALESCE(...) AS arpu_6m,
--               NTILE(5) OVER (ORDER BY jours_depuis_derniere_facture DESC) AS R_score,
--               NTILE(5) OVER (ORDER BY n_reclamations_6m DESC) AS F_score,
--               NTILE(5) OVER (ORDER BY arpu_6m) AS M_score
--        FROM clients c LEFT JOIN recence USING ... LEFT JOIN frequence USING ...
--             LEFT JOIN monetaire USING ...;
-- Puis : SELECT * FROM rfm LIMIT 10;




### 6.2 Attribution de segments RFM

#### 🔧 MÉTHODE

Règles métier classiques (à coder en `CASE WHEN`) :

- **Champions** : R≥4, F≥4, M≥4 (récents, peu de plaintes, ARPU élevé)
- **Loyal** : R≥3, F≥3, M≥3
- **At Risk** : R≤2 et M≥3 (n'ont pas facturé récemment mais étaient bons clients)
- **Lost** : R≤2 et M≤2
- **New** : R≥4, M≤2 (récent mais pas encore prouvé)
- **Others** : tous les autres

#### 🎯 Objectif — Cellule suivante

Pour chaque segment, calculer : nombre de clients, % de la base, ARPU 6m moyen, récence moyenne, plaintes 6m moyennes, **taux de churn**.

In [ ]:
%%sql
-- TODO : WITH segments AS (
--          SELECT *, CASE
--            WHEN R_score >= 4 AND F_score >= 4 AND M_score >= 4 THEN 'Champions'
--            WHEN R_score >= 3 AND F_score >= 3 AND M_score >= 3 THEN 'Loyal'
--            WHEN R_score <= 2 AND M_score >= 3 THEN 'At Risk'
--            WHEN R_score <= 2 AND M_score <= 2 THEN 'Lost'
--            WHEN R_score >= 4 AND M_score <= 2 THEN 'New'
--            ELSE 'Others' END AS segment_rfm
--          FROM rfm
--        )
--        SELECT segment_rfm, COUNT(*) AS n_clients,
--               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_base,
--               ROUND(AVG(arpu_6m), 0), ROUND(AVG(jours_depuis_derniere_facture), 0),
--               ROUND(AVG(n_reclamations_6m), 2),
--               ROUND(100.0 * COUNT(*) FILTER (WHERE statut = 'resilie') / COUNT(*), 1)
--                 AS taux_churn_pct
--        FROM segments GROUP BY segment_rfm ORDER BY n_clients DESC




### 6.3 Top 50 clients At Risk les plus prioritaires

#### 🔧 MÉTHODE

Au sein du segment *At Risk* (R≤2, M≥3), on classe par ARPU 6 mois descendant. Les premiers de la liste représentent **la valeur perdue prioritaire** : ce sont les clients qu'il faut appeler aujourd'hui.

#### 🎯 Objectif — Cellule suivante

Lister le **top 50** des clients At Risk (`R_score <= 2 AND M_score >= 3`) triés par `arpu_6m DESC`.

In [ ]:
%%sql
-- TODO : SELECT id_client, code_offre, ville, statut,
--               jours_depuis_derniere_facture, n_reclamations_6m,
--               ROUND(arpu_6m, 0) AS arpu_6m,
--               R_score, F_score, M_score
--        FROM rfm WHERE R_score <= 2 AND M_score >= 3
--        ORDER BY arpu_6m DESC LIMIT 50




### 🤔 Questions de réflexion

1. **Quel est le pourcentage de Champions** dans la base ? Une bonne base devrait avoir 15-25 % — IvoirCom est-il en deçà ou au-dessus ?
2. **Quelle est la part du segment Lost** ? Si > 20 %, c'est un signal d'usure structurelle.
3. **Le segment At Risk** : combien de clients, quel ARPU moyen, quel taux de churn observé ? Ces clients sont-ils prioritaires ?
4. **Top 50 At Risk** : sont-ils sur la même offre ? Si oui, laquelle ? Quelle action commerciale lancer dès demain matin ?

> 🏥 **MÉTIER** — Une bonne base devrait avoir 15-25 % de Champions et < 10 % d'At Risk. Si l'At Risk dépasse 15 %, on a une attrition de valeur élevée à venir.

---
## 7. Synthèse exécutive et 3 recommandations

### 🎯 À TOI DE JOUER — Synthèse pour la direction commerciale

Tu vas maintenant rédiger toi-même la **synthèse exécutive** dans une cellule markdown que tu ajoutes ci-dessous. Elle doit contenir :

**1. Tableau de bord récapitulatif** (à compléter avec les vrais chiffres)

| Indicateur | Valeur | Cible interne | Statut |
|---|---|---|---|
| Taux de churn cumulatif | ___ % | < 20 % | 🔴 / 🟠 / 🟢 |
| ARPU mensuel actifs | ___ FCFA | > 8 000 FCFA | 🔴 / 🟠 / 🟢 |
| Écart ARPU actifs vs churners | +___ % | — | indicatif |
| Taux churn offre la plus à risque | ___ % | < 25 % | 🔴 / 🟠 / 🟢 |
| Rétention M+11 cohorte janvier 2024 | ___ % | > 80 % | 🔴 / 🟠 / 🟢 |
| Taux churn parmi 2+ tickets non résolus | ___ % | < 30 % | 🔴 / 🟠 / 🟢 |
| Part At Risk dans la base | ___ % | < 10 % | 🔴 / 🟠 / 🟢 |
| Part Champions dans la base | ___ % | > 15 % | 🔴 / 🟠 / 🟢 |

**2. Diagnostic global en 2-3 phrases**

Que dit l'ensemble des chiffres sur la santé de la base IvoirCom ? Est-ce un problème d'**onboarding** (premiers mois) ou de **fidélisation moyen-long terme** ?

**3. Trois recommandations chiffrées prioritaires**

Pour chacune :
- Le **levier** (ce qu'on fait concrètement)
- La **mécanique** (combien de clients ciblés, quel taux de récupération attendu)
- L'**ARPU défendu / récupéré sur 1 an** (en M FCFA)

*Idées de leviers à explorer (non exhaustif)* :
- Cellule rétention basée sur le signal « 2+ tickets non résolus »
- Plan de migration depuis l'offre la plus churnante vers une offre supérieure
- Programme At Risk Top 50 avec appel commercial sortant

*(Cellule pour ta synthèse — double-clique pour la transformer en markdown et écris ta réponse ici)*

---
<div style="background:#1E3A5F;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">Customer Churn Analytics — IvoirCom</div>
<div style="font-size:13px;color:#CBD5E0;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>